# Dynamic training with the SoftStairs weight-type tracker

This notebook reuses the training pipeline from `experiments/CV/inception_stl10.ipynb` and integrates the
on-the-fly weight-type tracker (`softstairs_qat.analysis.weight_types`) into training, plus a sparse-support
experiment that compares optimization strategies on top of the tracker:

| mode | support receiving optimizer updates |
|---|---|
| `dense` | all quantized weights (forked dense baseline) |
| `dynamic_active` | tracker active set, recomputed every `support_update_interval` steps |
| `persistent_core` | discovery-time core: `time_active >= core-threshold` |
| `core_churning` | core \| churning, churning = `churning-min <= time_active < core-threshold` |
| `random_core` | fixed random support with `\|R\| = \|core\|` |
| `random_active` | fixed random support with `\|R\| =` median dynamic-active support size |

Protocol: dense QAT **discovery phase** (`discovery-fraction` of the epochs) while the tracker observes the
unmasked gradients; core/churning masks are frozen and everything (weights, tracker lifetime state, quantizer
t schedule, seeds, data ordering) forks from the same discovery checkpoint; every mode then trains the remaining
epochs with only its support unmasked (`grad *= mask` before `optimizer.step()`).

The experiment machinery lives in `experiments/sparse_support_experiment.py` (imported below); it can also be
run head-less via the CLI:
`python experiments/sparse_support_experiment.py --epochs 30 --modes dense dynamic_active ...`

Note: the tracker adds roughly 130 bytes/weight of GPU state. Reduce `batch_size` or pass `track_exclude`
(regex over parameter names) on memory-limited GPUs.

In [1]:
import os

os.chdir('../..')

In [2]:
DATASET='STL10'

In [3]:
import torch
import torchvision.models as models
import torch.nn as nn

def get_inception(n_classes):
    model = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)

    # Modify the final layer for STL-10 (10 classes)
    model.fc = nn.Linear(model.fc.in_features, 10)

    # If using auxiliary classifier (during training)
    model.aux_logits = True  # Set to False for inference
    if model.aux_logits:
        model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, n_classes)

    # Move to device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    return model 

# For STL-10 images are 96x96, Inception expects 299x299
# You'll need to resize your images
from torchvision import transforms

inception_transform = transforms.Compose([
    transforms.Resize((299, 299)),  # Inception expects 299x299
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Update your dataset transforms
test_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop((299, 299), scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
DATASET_PATH = 'experiments/SimCLR/data'
from torchvision.datasets import STL10

In [5]:
import pytorch_lightning as pl 
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_dataset = STL10(root=DATASET_PATH, split='train', transform=train_transform, download=False)
val_dataset = STL10(root=DATASET_PATH, split='train', transform=test_transform, download=False)
pl.seed_everything(42)
train_set, _ = torch.utils.data.random_split(train_dataset, [4000, 1000])
pl.seed_everything(42)
_, val_set = torch.utils.data.random_split(val_dataset, [4000, 1000])
test_set = STL10(root=DATASET_PATH,  split='test', transform=test_transform, download=True)

# We define a set of data loaders that we can use for various purposes later.
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, pin_memory=True, num_workers=4)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=4)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=4)

Seed set to 42
Seed set to 42


In [6]:
import torch

def reset_specific_gpu(gpu_id=0):
    """Reset a specific GPU."""
    if torch.cuda.is_available():
        with torch.cuda.device(gpu_id):
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
            print(f"GPU {gpu_id} reset")
            print(f"Memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

reset_specific_gpu(0)

GPU 0 reset
Memory allocated: 0.00 GB


In [7]:
example_batch = next(iter(train_loader))

In [8]:
from functools import partial
from softstairs_qat import SoftStairsQuantizer, QuantizationConfig


class SSQStep(pl.Callback):           
        
    def on_train_epoch_start(self, trainer, pl_module):
        epoch = trainer.current_epoch
        if not pl_module.quantizer:
            return
        if epoch > 0:
            pl_module.quantizer.step()
        t_value = pl_module.quantizer.t
        pl_module.log('quantizer_t', t_value, on_epoch=True, on_step=False)
        pl_module.log('quantizer_tback', pl_module.quantizer.current_backward_t, on_step=False)

    def on_validation_epoch_start(self, trainer, pl_module):
        if pl_module.quantizer:
            t_value = pl_module.quantizer.t
            pl_module.log('quantizer_t', t_value, on_epoch=True, on_step=False)
            pl_module.log('quantizer_tback', pl_module.quantizer.current_backward_t, on_step=False)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [9]:
import torch
from torch.ao.quantization import QConfig
from torch.ao.quantization.fake_quantize import FakeQuantize
from torch.ao.quantization.observer import HistogramObserver, PlaceholderObserver

def get_weight_only_qconfig(bit_width=8):
    """QConfig that only quantizes weights"""
    
    if bit_width == 8:
        # Int8 weights only
        weight_fake_quant = FakeQuantize.with_args(
            observer=HistogramObserver,
            quant_min=-128,
            quant_max=127,
            dtype=torch.qint8,
            qscheme=torch.per_tensor_affine,
        )
    elif bit_width == 4:
        # Int4 weights only
        weight_fake_quant = FakeQuantize.with_args(
            observer=HistogramObserver,
            quant_min=-8,
            quant_max=7,
            dtype=torch.qint8,
            qscheme=torch.per_tensor_affine,
        )
    else:
        raise ValueError(f"Unsupported bit width: {bit_width}")
    
    # Activations use PlaceholderObserver (no quantization)
    activation_fake_quant = PlaceholderObserver.with_args(
        dtype=torch.float32,
        is_dynamic=False,
    )
    
    return QConfig(
        activation=activation_fake_quant,
        weight=weight_fake_quant
    )

qconfig_weight_only = get_weight_only_qconfig(8)  # int8 weights
qconfig_weight_only_int4 = get_weight_only_qconfig(4)  # int4 weights

In [10]:
import torchmetrics
import torch.nn as nn
import torch.optim as optim
import pytorch_lightning as pl
from torchmetrics import MetricCollection
from torchmetrics.classification import Accuracy, F1Score, ConfusionMatrix
from torchmetrics import MeanMetric


class STL10Module(pl.LightningModule):
    def __init__(self, model_name, model_hparams, optimizer_name, optimizer_hparams, fake_quant=False, torch_qat=False, quantization_hparams={}):
        """STL10Module for InceptionNet."""
        super().__init__()
        self.save_hyperparameters()

        # Create model
        n_classes = self.hparams.model_hparams['n_classes']
        self.model = get_inception(n_classes)
        
        # Inception v3 specific: Set aux_logits based on training mode
        if hasattr(self.model, 'aux_logits'):
            self.model.aux_logits = True
        
        self.torch_qat = torch_qat
        self.fake_quant = fake_quant

        self._prepare_quantization()
        
        # Create loss module
        self.loss_module = nn.CrossEntropyLoss()
        
        # Initialize torchmetrics - Focus on Accuracy and F1
        self.train_metrics = MetricCollection({
            'mtr/acc': Accuracy(task="multiclass", num_classes=n_classes),
            'mtr/f1': F1Score(task="multiclass", num_classes=n_classes, average='macro'),
        })
        
        self.val_metrics = MetricCollection({
            'mtr/acc': Accuracy(task="multiclass", num_classes=n_classes),
            'mtr/f1': F1Score(task="multiclass", num_classes=n_classes, average='macro'),
        })
        
        self.test_metrics = MetricCollection({
            'mtr/acc': Accuracy(task="multiclass", num_classes=n_classes),
            'mtr/f1': F1Score(task="multiclass", num_classes=n_classes, average='macro'),
        })

        if self.fake_quant:
            self.val_metrics_quant = MetricCollection({
            'mtr/acc_quant': Accuracy(task="multiclass", num_classes=n_classes),
            'mtr/f1_quant': F1Score(task="multiclass", num_classes=n_classes, average='macro'),
        })
        
        self.val_confusion = ConfusionMatrix(task="multiclass", num_classes=n_classes)
        self.test_confusion = ConfusionMatrix(task="multiclass", num_classes=n_classes)
        # Example input for visualizing the graph in Tensorboard
        self.example_input_array = torch.zeros((1, 3, 299, 299), dtype=torch.float32)

    def _prepare_quantization(self):
        self.quantizer = None
        if self.torch_qat:
            qconfig = get_weight_only_qconfig(N_BITS)
            self.model.qconfig = qconfig
            from torch.quantization import prepare_qat, convert, get_default_qat_qconfig
            prepare_qat(self.model, inplace=True)
            
        else:
            excluded_modules = {
                n for n, p in self.model.named_modules()
                if "bn" in n or "aux" in n.lower()
            }
            
            qconfig = QuantizationConfig(**self.hparams.quantization_hparams)
            self.quantizer = SoftStairsQuantizer(
                self.model,
                qconfig,
                excluded_modules=excluded_modules,
            )

    def forward(self, imgs):
        if self.training and hasattr(self.model, 'aux_logits') and self.model.aux_logits:
            output, aux_output = self.model(imgs)
            return output, aux_output
        else:
            return self.model(imgs)

    def configure_optimizers(self):
        if self.hparams.optimizer_name == "Adam":
            optimizer = optim.AdamW(self.parameters(), **self.hparams.optimizer_hparams)
        elif self.hparams.optimizer_name == "SGD":
            optimizer = optim.SGD(self.parameters(), **self.hparams.optimizer_hparams)
        else:
            assert False, f'Unknown optimizer: "{self.hparams.optimizer_name}"'

        return [optimizer], []

    def training_step(self, batch, batch_idx):
        imgs, labels = batch
        
        if hasattr(self.model, 'aux_logits') and self.model.aux_logits:
            logits, aux_logits = self.model(imgs)
            loss1 = self.loss_module(logits, labels)
            loss2 = self.loss_module(aux_logits, labels)
            aux_loss_coef = .4
            loss = loss1 + aux_loss_coef * loss2
        else:
            logits = self.model(imgs)
            loss = self.loss_module(logits, labels)
        
        # Update training metrics
        preds = logits.argmax(-1)
        self.train_metrics.update(preds, labels)
        
        # Log metrics
        self.log_dict(self.train_metrics, on_step=True, on_epoch=True, prog_bar=True)
        self.log('loss/train', loss, on_step=True, on_epoch=True, prog_bar=False)
                
        return loss
    
    def on_train_epoch_start(self):
        if self.quantizer:
            self.log('epoch_start_quant_error', self.quantizer.estimate_current_quant_error(), on_epoch=True, prog_bar=True)
    

    def on_validation_epoch_end(self):        
        # Reset metrics for next epoch
        self.val_metrics.reset()
        self.val_metrics_quant.reset()
        torch.cuda.empty_cache()
        import gc 
        gc.collect()


    def validation_step(self, batch, batch_idx):
        imgs, labels = batch
        
        # Disable aux_logits for validation
        if hasattr(self.model, 'aux_logits'):
            self.model.aux_logits = False
        
        logits = self.model(imgs)
        loss = self.loss_module(logits, labels)

        preds = logits.argmax(-1)
        
        # Update validation metrics
        self.val_metrics.update(preds, labels)
        
        # Log validation metrics - focus on accuracy and F1
        self.log_dict(self.val_metrics, on_step=True, on_epoch=True, prog_bar=True)
        self.log("loss/val", loss, on_epoch=True, on_step=True)
        
        # Re-enable aux_logits
        if self.training and hasattr(self.model, 'aux_logits'):
            self.model.aux_logits = True
            
        
        if not (self.fake_quant and self.quantizer):
            return loss 
        
        # save original state
        
        state_dict = {k: v.clone() for k, v in self.model.state_dict().items()}
        self.quantizer.fake_quantize()


        logits = self.model(imgs)
        loss_quant = self.loss_module(logits, labels)

        preds = logits.argmax(-1)
        
        # Update validation metrics
        self.val_metrics_quant.update(preds, labels)
        
        self.log_dict(self.val_metrics_quant, on_step=True, on_epoch=True, prog_bar=True)
        self.log("loss/val_quant", loss_quant, on_epoch=True, on_step=True)

        self.model.load_state_dict(state_dict)
        del state_dict
        self.quantizer.activate_hooks()
        
        
        return loss



    def test_step(self, batch, batch_idx):
        imgs, labels = batch
        
        # Disable aux_logits for testing
        if hasattr(self.model, 'aux_logits'):
            self.model.aux_logits = False
            
        logits = self.model(imgs)
        preds = logits.argmax(-1)
        
        # Update test metrics
        self.test_metrics.update(logits, labels)
        
        # Log test metrics
        self.log_dict(self.test_metrics, on_step=True, on_epoch=True)

    def on_test_epoch_end(self):
        # Reset metrics
        self.test_metrics.reset()

    def on_train_epoch_end(self):
        # Reset training metrics at the end of each epoch
        if self.quantizer:
            self.log('epoch_end_quant_error', self.quantizer.estimate_current_quant_error(), on_epoch=True, prog_bar=True)
        self.train_metrics.reset()

In [11]:
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint  # noqa: E402/

In [12]:
CHECKPOINT_PATH = os.environ.get("PATH_CHECKPOINT", "saved_models/ConvNets")

In [13]:
import os
import torch
from pytorch_lightning.callbacks import Callback


class GradSaver(Callback):
    """
    Save up to `P` gradient snapshots per epoch, at optimizer steps.

    Within each epoch, the first `P` optimizer steps trigger a save
    (grad + parameter snapshot). The counter resets at the start of every
    epoch, so each epoch gets its own up-to-P samples.
    """

    def __init__(self, parameter_name: str, P: int = 10):
        self.parameter_name = parameter_name
        self.save_dir = parameter_name
        self.P = int(P)

        self.hook = None
        self.grad = None
        self.parameter = None
        self.log_dir = None

        # per-epoch state
        self._epoch_saved = 0
        self._epoch = 0

    # ------------------------------------------------------------------ setup

    def on_fit_start(self, trainer, pl_module):
        part = pl_module.model
        for name in self.parameter_name.split('.'):
            part = getattr(part, name)
        parameter: torch.nn.Parameter = part
        self.parameter = parameter
        parameter.requires_grad_(True)

        def gather_grad(param):
            self.grad = param.grad

        self.hook = parameter.register_post_accumulate_grad_hook(gather_grad)

    # ---------------------------------------------------------- epoch hooks

    def on_train_epoch_start(self, trainer, pl_module):
        self._epoch_saved = 0
        self._epoch = pl_module.current_epoch

    # ------------------------------------------------------- optimizer steps

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        if self.hook is None or self.grad is None:
            return
        if self._epoch_saved >= self.P:
            return

        self._save_snapshot(trainer, pl_module)
        self._epoch_saved += 1

    # -------------------------------------------------------------- saving

    def _ensure_log_dir(self, trainer):
        if self.log_dir is None:
            base = trainer.log_dir or trainer.default_root_dir
            self.log_dir = os.path.join(base, self.save_dir)
            os.makedirs(self.log_dir, exist_ok=True)

    def _save_snapshot(self, trainer, pl_module):
        self._ensure_log_dir(trainer)

        # Identify the snapshot: epoch + index-within-epoch.
        tag = f'{self._epoch}.{self._epoch_saved}'

        grad_tensor = self.grad.detach().float().cpu()
        torch.save(grad_tensor, os.path.join(self.log_dir, f'grad_{tag}.pth'))

        if getattr(pl_module, 'quantizer', None) is not None:
            param = pl_module.quantizer.upscaled_parameter(
                self.parameter_name + pl_module.quantizer._orig_suffix
            )
            torch.save(param.detach().float().cpu(),
                       os.path.join(self.log_dir, f'param_{tag}.pth'))

    # --------------------------------------------------------------- teardown

    def on_fit_end(self, trainer, pl_module):
        if self.hook:
            self.hook.remove()
            self.hook = None
        self.grad = None

In [14]:
from pytorch_lightning.loggers import CSVLogger 

def train_model(model_name, save_name=None, epochs=100, torch_qat=False, quantization_hparams={}, retrain=False, **kwargs):
    """Train model.

    Args:
        model_name: Name of the model you want to run. Is used to look up the class in "model_dict"
        save_name (optional): If specified, this name will be used for creating the checkpoint and logging directory.

    """
    if save_name is None:
        save_name = model_name


    logger = CSVLogger(
        save_dir="experiments/CV/logs",
        name=save_name,
    )

    # Create a PyTorch Lightning trainer with the generation callback
    trainer = pl.Trainer(
        default_root_dir=os.path.join(CHECKPOINT_PATH, save_name),  # Where to save models
        accelerator="auto",
        devices=1,
        max_epochs=epochs,
        callbacks=[
            SSQStep(),
            LearningRateMonitor("epoch"),
            GradSaver('fc.weight', 20, ),
            GradSaver('Mixed_6e.branch1x1.conv.weight', 20),
          ],  # Log learning rate every epoch
        logger=logger,

    )
    trainer.logger._default_hp_metric = None  # Optional logging argument that we don't need

    # Check whether pretrained model exists. If yes, load it and skip training
    pretrained_filename = os.path.join(CHECKPOINT_PATH, save_name + ".ckpt")
    if not retrain and os.path.isfile(pretrained_filename):
        print(f"Found pretrained model at {pretrained_filename}, loading...")
        model = STL10Module.load_from_checkpoint(pretrained_filename, quantization_hparams=quantization_hparams | {'steps': epochs})
    else:
        pl.seed_everything(42)  # To be reproducible
        model = STL10Module(model_name=model_name, quantization_hparams=quantization_hparams  | {'n_steps': epochs}, torch_qat=torch_qat,  **kwargs)
        trainer.fit(model, train_loader, val_loader)

    return model

## Dynamic training: tracker + sparse-support scheduling

The cells below drive the full experiment through `sparse_support_experiment.run_experiment`, which wires the
reused pipeline to two callbacks:

- `SparseSupportController` builds a `ModelWeightTypeTracker` over the quantized weight leaves (`*_orig`) at
  fit start, feeds it the **unmasked** QAT gradients every optimizer step (`wt/*` metrics in the CSV logs),
  and zeroes optimizer gradients outside the mode's support mask before `optimizer.step()` (`sp/*` metrics:
  support fraction, update budget, `J(active, core)`, core/churning coverage, support-change rate).
- `SSQStepCallback` steps the t schedule once per epoch, resume-aware (forked runs are fast-forwarded to the
  discovery temperature beforehand).

Discovery artifacts land in `<results-dir>/discovery/`: `discovery.ckpt`, `masks.pt` (core / churning /
`time_active`), `tracker_state.pt`, `weight_stats.pt` (per-weight export).

In [15]:
import sys

sys.path.insert(0, 'experiments')

from sparse_support_experiment import (
    MODES,
    MODE_DESCRIPTIONS,
    SSQStepCallback,
    SparseSupportController,
    build_tracker_config,
    default_args,
    run_experiment,
)
from softstairs_qat.analysis import ModelWeightTypeTracker, WeightType, WeightTypeConfig

In [22]:
EXPERIMENT_KWARGS = dict(
    results_dir='experiments/CV/logs/dynamic_training',
    epochs=30,               # total epochs: 25% discovery, 75% optimization
    discovery_fraction=0.34,
    batch_size=16,
    modes=list(MODES),
)

args = default_args(**EXPERIMENT_KWARGS)
summary = run_experiment(args)
summary

Seed set to 42
Seed set to 42


[experiment] discovery: 10/30 epochs | modes: ('dense', 'dynamic_active', 'persistent_core', 'core_churning', 'random_core', 'random_active')
[experiment] quantization: {'t_scheduler_strategy': 'cyclic', 'async_t_factor': 1.0, 't_start': 0.9, 't_end': 0.0005, 't_tau': 8.0, 'n_bits': 8, 'type': 'standard', 'normalized': True, 'symmetric': False, 'early_power': 0.95, 'min_majorant': 0.0, 'half_shift': False, 'n_steps': 30}


Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name              ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model             │ Inception3       │ 24.4 M │ train │     0 │
│ 1 │ loss_module       │ CrossEntropyLoss │      0 │ train │     0 │
│ 2 │ train_metrics     │ MetricCollection │      0 │ train │     0 │
│ 3 │ val_metrics       │ MetricCollection │      0 │ train │     0 │
│ 4 │ val_metrics_quant │ MetricCollection │      0 │ train │     0 │
└───┴───────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 24.4 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.4 M                                                                                               
Total estimated model params size (MB): 97.486                                                                     
Modules in train mode: 317                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/leostre/Рабочий стол/SoftStairs-QAT/.venv/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/leostre/Рабочий стол/SoftStairs-QAT/.venv/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: The ``compute`` method of metric MulticlassAccuracy was called before the ``update`` method which may lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)
/home/leostre/Рабочий стол/SoftStairs-QAT/.venv/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: The ``compute`` method of metric MulticlassF1Score was called before the ``update`` method which may lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)


OutOfMemoryError: CUDA out of memory. Tried to allocate 168.00 MiB. GPU 0 has a total capacity of 15.46 GiB of which 109.88 MiB is free. Including non-PyTorch memory, this process has 3.78 GiB memory in use. Process 79132 has 10.08 GiB memory in use. Of the allocated memory 3.05 GiB is allocated by PyTorch, and 404.86 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [17]:
from pathlib import Path 
import pandas as pd 


def read_results(name, scale='step', version=None, agg_func='last'):
    path = Path(f'experiments/CV/logs/{name}')
    if not version:
        version = sorted(os.listdir(path), key=lambda x: float(x.split('_')[-1]))[-1]
    elif isinstance(version, int):
        version = f'version_{version}'
    file = path / version / 'metrics.csv' 
    df = pd.read_csv(file)

    df.ffill(inplace=True)
    df = df.groupby(scale).aggregate(agg_func).reset_index()
    alternative_scale = 'step' if scale != 'step' else 'epoch'
    df.drop([c for c in df.columns if alternative_scale in c], axis=1, inplace=True)
    return df, 0.

In [21]:
time_active.size()

torch.Size([21771616])

In [18]:
import torch

discovery_dir = Path(EXPERIMENT_KWARGS['results_dir']) / 'discovery'
masks = torch.load(discovery_dir / 'masks.pt', map_location='cpu', weights_only=False)
stats = torch.load(discovery_dir / 'weight_stats.pt', map_location='cpu', weights_only=False)
time_active = masks['time_active']

print(f"tracked weights: {masks['num_tracked_weights']}")
print(f"core (time_active >= {masks['core_threshold']}): {int(masks['core'].sum())}")
print(f"churning ({masks['churning_min']} <= time_active < {masks['core_threshold']}): {int(masks['churning'].sum())}")
print('long-term types:', {WeightType(kind).name: int((stats['long_term_type'] == kind).sum()) for kind in sorted(torch.unique(stats['long_term_type']).tolist())})
print('quantiles of time_active:', torch.quantile(time_active, torch.tensor([0.25, 0.5, 0.75])).tolist())

tracked weights: 21771616
core (time_active >= 0.9): 40226
churning (0.1 <= time_active < 0.9): 7215377
long-term types: {'NEVER_ACTIVE': 3490014, 'RARE_ACTIVE': 11025999, 'CHURNING': 7215377, 'PERSISTENT_ACTIVE': 40226}


RuntimeError: quantile() input tensor is too large

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")
sns.set_palette("Set2")

fig, axes = plt.subplots(1, 3, figsize=(21, 4.5))
axes[0].hist(time_active.numpy(), bins=50, color='steelblue')
axes[0].axvline(float(masks['core_threshold']), color='crimson', linestyle='--', label='core threshold')
axes[0].axvline(float(masks['churning_min']), color='darkorange', linestyle='--', label='churning min')
axes[0].set_title('discovery time_active')
axes[0].legend()

for mode in summary['mode']:
    df, _ = read_results(f"dynamic_training/{mode}", scale='step', agg_func='mean')
    if 'sp/support_fraction_step' in df:
        sns.lineplot(df, x='step', y='sp/support_fraction_step', ax=axes[1], label=mode)
    if 'sp/jaccard_active_core_step' in df:
        sns.lineplot(df, x='step', y='sp/jaccard_active_core_step', ax=axes[2], label=mode)
axes[1].set_title('support fraction per optimizer step')
axes[2].set_title('J(tracker active set, core)')
plt.tight_layout()

In [ ]:
def plot_dynamics(name, scale='step', metric='acc', axes=None, suffix='', agg_func='mean'):
    name, *version = name.split('#')
    if not version:
        version = None
    else:
        version = int(version[0])
    dynamics, final_results = read_results(name, scale, version, agg_func)
    
    if axes is None: 
        fig, axes = plt.subplots(2, 2, figsize=(20, 20))

    # losses
    for c in [c for c in dynamics.columns if c.startswith('loss/')]:
        sns.lineplot(dynamics, x=scale, y=c, ax=axes[0][0], label=c + suffix)
    axes[0][0].legend()
    axes[0][0].set_title('losses') 

    # mtr
    for c in [c for c in dynamics.columns if not metric and c.startswith('mtr/') or isinstance(metric, str) and metric in c]:
        sns.lineplot(dynamics, x=scale, y=c, ax=axes[0][1], label=c + suffix)
    axes[0][1].legend()
    axes[0][1].set_title('metrics') 

    if not 'quantizer_t' in dynamics:
        return axes

    # t schedule 
    ax = axes[1][0] 
    sns.lineplot(dynamics, x=scale, y='quantizer_t', ax=ax, label='t' + suffix)
    tback = 'quantizer_tback'
    if tback in dynamics.columns:
        sns.lineplot(dynamics, x=scale, y=tback, ax=ax, label=tback + suffix)
    ax.set_title(f't_schedule ')
    

    # metric vs. t_schedule 
    ax = axes[1][1]
    for c in [c for c in dynamics.columns if not metric and c.startswith('mtr/') or isinstance(metric, str) and metric in c]:
        sns.scatterplot(dynamics, x='quantizer_t', y=c, ax=ax, label=c + suffix)
        if not metric:
            break
    ax.set_title(f'{metric} vs. t_schedule ')
    return axes 


def plot_comparison(*exps, scale, metric, suffixes=None, agg_func='mean'):
    if suffixes is None:
        suffixes = [f" ({'-'.join(exp.split('_')[2:])})" for exp in exps]
    axes = plot_dynamics(exps[0], scale, metric, agg_func=agg_func, suffix=suffixes[0])
    for exp, suffix in zip(exps[1:], suffixes[1:]):
        axes = plot_dynamics(exp, scale, metric, axes, suffix, agg_func=agg_func)
    return axes


plot_comparison(*[f'dynamic_training/{mode}' for mode in summary['mode']], scale='epoch', metric='acc', agg_func='last')

In [19]:
summary[[
    'mode',
    'test_acc',
    'test_acc_quant',
    'best_val_acc_quant',
    'support_fraction_mean',
    'support_size_final',
    'update_budget_fraction',
    'effective_updates',
    'jaccard_active_core_mean',
    'core_coverage_mean',
    'churning_coverage_mean',
    'support_change_rate_mean',
    'acc_drop_vs_dense',
]]

,mode,test_acc,test_acc_quant,best_val_acc_quant,support_fraction_mean,support_size_final,update_budget_fraction,effective_updates,jaccard_active_core_mean,core_coverage_mean,churning_coverage_mean,support_change_rate_mean,acc_drop_vs_dense
0,dense,0.863500,0.863500,0.854,1.000000,NaN,0.666667,18604200983,0.011671,0.818059,0.298675,NaN,0.000000
1,dynamic_active,0.862250,0.862250,0.858,0.129327,2817638.0,0.086218,2815660352,0.011783,0.826774,0.302933,0.694536,0.001250
2,persistent_core,0.847750,0.847750,0.837,0.001848,40226.0,0.001232,40214989,0.012107,0.870059,0.311411,NaN,0.015750
3,core_churning,0.858375,0.858375,0.861,0.333260,7255603.0,0.222173,7088186853,0.011892,0.830370,0.327079,NaN,0.005125
4,random_core,0.848625,0.848625,0.828,0.001848,40226.0,0.001232,34334827,0.011873,0.854053,0.310152,NaN,0.014875
5,random_active,0.854625,0.854625,0.840,0.129181,2812485.0,0.086121,2406485824,0.011744,0.840716,0.307188,NaN,0.008875
